# HW5: Fixed-Income Valuation and First-Order Risk

Complete the five library TODOs, then run all cells.

## Part I: Implicit Function Differentiation

**Setting.** A parametric model $M(X_I)$ has internal state $X_I\in\mathbb{R}^n$ (for a yield curve: the piecewise-constant IFR levels). A vector of quote functions $V=(V_1,\dots,V_n)$ maps the state to the model-implied value of each market observable, $V_i(X_I)=X^M_i$, and the market supplies $X_M\in\mathbb{R}^n$. Since $\dim X_I=\dim X_M=n$ the system is square.

### Q1. General recipe for solving $V(X_I)=X_M$

Calibration is a root-finding problem for the residual $F(X_I):=V(X_I)-X_M=0$:

1. **Parameterise and initialise.** Fix the interpolation/extrapolation scheme so that the curve is fully described by the finite vector $X_I$, and pick a starting point $X_I^{(0)}$ (e.g. flat at the first quote, or yesterday's state).
2. **Residual and Jacobian.** Evaluate $F(X_I)$ and the $n\times n$ model Jacobian $J_{V/X_I}=\partial V/\partial X_I$, whose $i$-th row is $\nabla_{X_I}V_i$. The rows come from the chain rule through the building blocks, $V_i\Rightarrow\{P(0,T),\,R(0;T,T')\}\Rightarrow f(\cdot)\Rightarrow X_I$: exactly the discount-factor gradients implemented in TODO 1-2 (an analytic Jacobian is preferred; bump-and-reprice is the fallback and the check).
3. **Newton-Raphson.** Solve $J\,\Delta X=-F(X_I^{(k)})$ and set $X_I^{(k+1)}=X_I^{(k)}+\Delta X$ (with damping or a line search if the first steps are poor); stop when $\lVert F\rVert<\text{tol}$ and the inputs reprice.
4. **Exploit the structure.** Each calibration instrument is only sensitive to the state up to its anchor point. Ordering instruments by anchor point makes $J$ lower-triangular, so the $n$-dimensional Newton solve collapses into $n$ sequential one-dimensional root solves (bootstrap): $x_1$ from $V_1$, then $x_2$ from $V_2$ given $x_1$, and so on (1-D Newton, Brent, or bisection per node).
5. **Well-posedness checks.** $J$ must be non-singular at the solution (distinct anchor points, one new state variable per instrument); otherwise the problem is under/over-determined and the implicit-function argument below fails.

### Q2. Computing $\dfrac{dX_I}{dX_M}$ at the optimum

At the calibrated point the identity $V\big(X_I^*(X_M)\big)=X_M$ holds for every $X_M$ in a neighbourhood of today's market. Differentiating both sides with respect to $X_M$ (chain rule) gives

$$
\frac{\partial V}{\partial X_I}\Big|_{X_I^*}\;\frac{dX_I^*}{dX_M}=I
\qquad\Longrightarrow\qquad
\frac{dX_I^*}{dX_M}=\Big(\frac{\partial V}{\partial X_I}\Big)^{-1}=J_{V/X_I}^{-1}.
$$

This is the implicit function theorem: it requires only that the Jacobian already used by Newton be invertible at the solution, and it needs **no** extra re-calibration (no bumping each quote and re-solving). In the bootstrap case $J$ is triangular, so the inverse is a cheap back-substitution.

The main use is to convert *state* risk into *market-quote* risk. If a product has value $PV(X_I)$ with raw state gradient $\nabla_{X_I}PV$ (what `calculate_risk` produces), then

$$
\nabla_{X_M}PV=\Big(\frac{dX_I}{dX_M}\Big)^{\!\top}\nabla_{X_I}PV=J_{V/X_I}^{-\top}\,\nabla_{X_I}PV ,
$$

i.e. the PV change per 1 bp of each quoted swap/future rate, which is what the desk hedges with. In this notebook the market data *are* the IFR nodes, so $V$ is the identity, $J=I$, and the state gradient is already the quote gradient (that is why `RiskReport` applies no Jacobian).

### Q3. Application to the implied-vol interpolator

* **Observables and state.** $X_M=(P_1,P_2,P_3)$ are undiscounted OTM prices at $K_1<K_{atm}<K_3$ (a put at $K_1$, the ATM option at $K_{atm}$, a call at $K_3$); undiscounted means the Black-76 forward price with the discount factor stripped out. The internal state is $X_I=\theta$, and because three prices are matched by three parameters the system is square. The quote functions are
  $$V_i(\theta)=\mathrm{Black}\big(F,K_i,\sigma(K_i;\theta),T,\omega_i\big),\qquad i=1,2,3 .$$
* **Solve for $\theta$ (Q1).** First invert each price into a market implied vol $\sigma_i^{mkt}$ (a 1-D root search; Black is monotone in $\sigma$, so this is unique). The calibration then becomes $\sigma(K_i;\theta)=\sigma_i^{mkt}$, $i=1,2,3$: three equations, three unknowns. Run Newton on $\theta$ with Jacobian $J_{ij}=\partial\sigma(K_i;\theta)/\partial\theta_j$ (or, directly in price space, $J_{ij}=\mathrm{vega}_i\cdot\partial\sigma(K_i;\theta)/\partial\theta_j$ by the chain rule). For a quadratic interpolator in log-moneyness $x=\ln(K/F)$, $\sigma(K)=\theta_0+\theta_1x+\theta_2x^2$, the system is linear and solves in one step; for a SABR-type parameterisation $(\alpha,\rho,\nu)$ it is a genuine $3\times3$ nonlinear Newton solve. Uniqueness holds when $J$ is invertible, i.e. the three strikes are distinct and the three parameters are identifiable.
* **Sensitivity of $\sigma(K)$ to $\theta$ (Q2).** For an arbitrary strike, $\partial\sigma(K)/\partial\theta$ is the interpolator's parameter derivative (for the quadratic: $(1,x,x^2)$; for SABR: derivatives of Hagan's formula). Combining with the implicit function theorem gives how the whole smile responds to one market price:
  $$\frac{d\sigma(K)}{dP_j}=\sum_k\frac{\partial\sigma(K)}{\partial\theta_k}\Big[J^{-1}\Big]_{kj},\qquad J=\frac{\partial V}{\partial\theta}.$$
  This is the vol-surface analogue of converting IFR-state risk into swap-rate risk.
* **What $\theta$ should mean to a trader.** The three strikes pin down the *level*, the *slope* and the *curvature* of the smile, so the natural trader basis is
  $$\theta\;\leftrightarrow\;(\text{ATM vol},\ \text{risk reversal},\ \text{butterfly}),\qquad RR=\sigma_{call}-\sigma_{put},\quad BF=\tfrac12(\sigma_{call}+\sigma_{put})-\sigma_{atm}.$$
  With that choice the sensitivities read "parallel shift of the smile per 1 vol point of ATM", "tilt per 1 vol point of RR" and "wing/convexity move per 1 vol point of BF", which are exactly the instruments (straddles, risk reversals, butterflies) the desk quotes and hedges with. If the interpolator is stored in some other parameterisation (e.g. SABR's $\alpha,\rho,\nu$ or polynomial coefficients), map it with one more Jacobian, $d\sigma/dq=(d\sigma/d\theta)\,(dq/d\theta)^{-1}$ for $q=(\text{ATM},RR,BF)$; intuitively $\alpha\leftrightarrow$ ATM level, $\rho\leftrightarrow$ skew/RR, $\nu\leftrightarrow$ smile/BF.

In [1]:
from typing import cast

import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfCreateData1D,
    qfCreateDataCollection,
    qfCreateBuildMethod,
    qfCreateModelBuildMethodCollection,
    qfCreateModel,
    qfCreateProductFixedAccruedCashflow,
    qfCreateValuationParameters,
    qfCreateValuationParametersCollection,
    qfCreateValueReport,
)
from fixedincomelib.date import Date
from fixedincomelib.market.registries import IndexRegistry
from fixedincomelib.valuation import ValuationRequest
from fixedincomelib.yield_curve import YieldCurveModelComponent
from fixedincomelib.yield_curve.valuation_engine import ValuationEngineProductFixedAccrued

np.set_printoptions(precision=6, suppress=True)
pd.options.display.float_format = "{:,.6f}".format
pd.options.display.max_colwidth = None

## 1. Input data

Rates are annual decimals: `0.03 = 3%`. Edit the inputs here and rerun the following cells.
The funding component adds an IFR spread over SOFR.

In [2]:
VALUE_DATE = "2026-01-02"
PAYMENT_DATE = Date("2026-10-02")
FUNDING_INDEX = "SOFR-1B-FLAT"

sofr_quotes = pd.DataFrame(
    {"values": [0.03, 0.04, 0.05]},
    index=pd.Index(["2026-04-02", "2027-01-02", "2028-01-02"], name="Node date"),
)
funding_quotes = pd.DataFrame(
    {"values": [0.0, 0.001]},
    index=pd.Index(["2026-07-02", "2027-01-02"], name="Node date"),
)

display(sofr_quotes.rename(columns={"values": "SOFR-1B IFR"}))
display(funding_quotes.rename(columns={"values": "SOFR-1B-FLAT IFR spread"}))

,SOFR-1B IFR
Node date,
2026-04-02,0.030000
2027-01-02,0.040000
2028-01-02,0.050000


,SOFR-1B-FLAT IFR spread
Node date,
2026-07-02,0.000000
2027-01-02,0.001000


## 2. Data and build methods

Create the IFR datasets and build methods, with SOFR first and funding second.

In [3]:
sofr_data = qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-IFR", sofr_quotes)
funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE", "SOFR-1B-FUNDING-IFR", funding_quotes,
)
data_collection = qfCreateDataCollection([sofr_data, funding_data])
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-IFR
1,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-FUNDING-IFR


In [4]:
sofr_method = qfCreateBuildMethod("YIELD_CURVE_INDEX", {
    "TARGET": "SOFR-1B", "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
})
funding_method = qfCreateBuildMethod("YIELD_CURVE_FUNDING", {
    "TARGET": FUNDING_INDEX, "REFERENCE INDEX": "SOFR-1B",
    "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
})
build_methods = qfCreateModelBuildMethodCollection([sofr_method, funding_method])

## 3. Build the model

Copy IFR inputs into the model. The registered funding component references SOFR.

In [5]:
model = qfCreateModel(VALUE_DATE, "YIELD_CURVE", data_collection, build_methods)
print("Component sizes:", model.num_sub_components)

Component sizes: [3, 2]


The following helpers retrieve a component (`retrieve_model_component`), allocate zero-filled
gradient blocks in component order (`resize_gradient`), and locate a component's block
(`retrieve_model_component_index`, using the `component_indices` map).

In [6]:
index = IndexRegistry().get("SOFR-1B")
component = cast(YieldCurveModelComponent, model.retrieve_model_component(index))
funding_component = model.retrieve_model_component(FUNDING_INDEX)

print("SOFR-1B ->", type(component).__name__, "|", component.component_identifier.name())
print("SOFR-1B-FLAT ->", funding_component.component_identifier.name())
display(pd.DataFrame(component.state_data.T, columns=["Node time (years)", "IFR"]))

SOFR-1B -> YieldCurveModelComponent | SOFRON Actual/360
SOFR-1B-FLAT -> SOFR-1B-FLAT


,Node time (years),IFR
0,0.246575,0.030000
1,1.000000,0.040000
2,1.999993,0.050000


In [7]:
gradient = []
model.resize_gradient(gradient)
print("[SOFR block, funding block]:", gradient)

[SOFR block, funding block]: [array([0., 0., 0.]), array([0., 0.])]


In [8]:
print("Stored map:", model.component_indices)
print("SOFR-1B -> block", model.retrieve_model_component_index("SOFR-1B"))
print("SOFR-1B-FLAT -> block", model.retrieve_model_component_index(FUNDING_INDEX))

Stored map: {'SOFRON Actual/360': 0, 'SOFR-1B-FLAT': 1}
SOFR-1B -> block 0
SOFR-1B-FLAT -> block 1


## 4. Discount-factor gradients

Complete **TODO 1** in `YieldCurveModelComponent` and **TODO 2** in `YieldCurve`
(`fixedincomelib/yield_curve/yield_curve_model.py`). Calculate the local derivative, then apply
the product rule recursively to $D_{\mathrm{funding}}=D_{\mathrm{spread}}D_{\mathrm{SOFR}}$.

Gradient methods write raw derivatives into the supplied container and return `None`.
`scaler` scales the new contribution; `accumulate=False` replaces the output and `True` adds to it.

With the default piecewise-constant interpolation, the third SOFR segment starts after
2027-01-02. Its gradient is zero for the 2026-10-02 payment.

In [9]:
local_gradient = np.zeros(component.num_state_data)
component.discount_factor_gradient_wrt_state(PAYMENT_DATE, local_gradient)
print("Local SOFR discount-factor gradient:", local_gradient)

Local SOFR discount-factor gradient: [-0.239898 -0.487793  0.      ]


In [10]:
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, gradient)
print("Model gradient [SOFR block, funding block]:", gradient)

Model gradient [SOFR block, funding block]: [array([-0.239838, -0.48767 ,  0.      ]), array([-0.482341, -0.245168])]


## 5. Fixed-accrued valuation engine

Complete **TODOs 3–5** in `ValuationEngineProductFixedAccrued`: `__init__`, `calculate_value`,
and `calculate_risk` (`fixedincomelib/yield_curve/valuation_engine.py`).
Select the funding curve, then compute $C=N\alpha$, $PV=C D(T_p)$ and
$\nabla PV=C\nabla D(T_p)$. Notional is signed; this product has no separate fixed-rate field.

| Payment date | PV / cash today | Risk |
|---|---|---|
| Future | $C D(T_p)$ / 0 | $C\nabla D(T_p)$ |
| Today | $C$ / $C$ | 0 |
| Past | 0 / 0 | 0 |

In [11]:
product = qfCreateProductFixedAccruedCashflow(
    "2026-04-02", "2026-07-02", "USD", 1_000_000, "ACT/360",
    payment_date=PAYMENT_DATE.ISO(), business_day_convention="NONE",
)
funding_parameter = qfCreateValuationParameters(
    "FUNDING INDEX PARAMETER", {"FUNDING INDEX": FUNDING_INDEX},
)
parameters = qfCreateValuationParametersCollection([funding_parameter])

print("Accrual fraction:", product.accrued)
print("Cash amount:", product.notional * product.accrued)

Accrual fraction: 0.25277777777777777
Cash amount: 252777.77777777778


In [12]:
engine = ValuationEngineProductFixedAccrued(model, parameters, product, ValuationRequest.PV)
engine.calculate_value()

product_gradient = []
engine.calculate_risk(product_gradient)

print("PV:", engine.value)
print("Cash today:", engine.cash)
print("Product gradient [SOFR block, funding block]:", product_gradient)

PV: 245870.91057967892
Cash today: 0.0
Product gradient [SOFR block, funding block]: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


## 6. Reports

`qfCreateValueReport` calls the engine for `PV` or `FirstOrderRisk`.
`state_gradient` retains raw blocks; `display()` multiplies risk by `0.0001`
to show the signed PV change per positive 1 bp node shift.

In [13]:
print("PV report:", qfCreateValueReport(model, product, parameters, "PV"))
risk_report = qfCreateValueReport(model, product, parameters, "FirstOrderRisk")

print("Raw blocks:", risk_report.state_gradient)
display(risk_report.display()[["COMPONENT", "NODE_INDEX", "AXIS1", "UNIT", "VALUES"]])

PV report: [['USD', 245870.91057967892]]
Raw blocks: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


,COMPONENT,NODE_INDEX,AXIS1,UNIT,VALUES
0,SOFRON Actual/360,0,2026-04-02,0.000100,-6.062570
1,SOFRON Actual/360,1,2027-01-02,0.000100,-12.327226
2,SOFRON Actual/360,2,2028-01-02,0.000100,0.000000
3,SOFR-1B-FLAT,0,2026-07-02,0.000100,-12.192503
4,SOFR-1B-FLAT,1,2027-01-02,0.000100,-6.197294
